In [ ]:
%matplotlib widget

In [ ]:
from pathlib import Path
import numpy as np
import tifffile as tiff
import flammkuchen as fl
import napari
from scipy.ndimage import map_coordinates

In [ ]:
from fimpylab.core.twop_experiment import TwoPExperiment

In [ ]:
from napari_animation import AnimationWidget

In [ ]:
%gui qt5

In [ ]:
def map_affine(stack,  transform_mat, dest_shape, order=3):
    target_coords = np.reshape(transform_mat @ np.reshape(
        np.pad(np.indices(dest_shape), ((0,1), (0,0), (0,0), (0,0)),
        mode="constant", constant_values=1), (4, -1)), 
                                (3, *dest_shape))

    return map_coordinates(stack, target_coords, order=order)

In [ ]:
master = Path(r"Z:\Hagar\development\dendra anatomy 24 hours intervals")
path_list = list(master.glob("*_f3"))
print(path_list)

In [ ]:
exp_path =  path_list[0]
exp = TwoPExperiment(exp_path)
res = exp.resolution

In [ ]:
ref_path = path_list[1] / "registration" 
ref = fl.load(ref_path / "ref_mapped.h5")
viewer = napari.view_image(ref.T,  colormap="gray_r")

In [ ]:
for f in path_list[1:]:
    #Apply transform
    path = f / "registration" 
    mov1 = fl.load(path / "mov_mapped.h5")
    mat1 = fl.load(path / "initial_transform_mapped.h5")
    for i in range(2):
        mat1[i,i] = 1
    transformed = map_affine(mov1, mat1, ref.shape)
    viewer.add_image(transformed.T, colormap="gray_r")

In [ ]:
path = f / "registration" 
mov1 = fl.load(path / "mov_mapped.h5")
mat1 = fl.load(path / "initial_transform_mapped.h5")
mat1

In [ ]:
animation_widget = AnimationWidget(viewer)
viewer.window.add_dock_widget(animation_widget, area='right')

In [ ]:
mat1

In [ ]:
viewer = napari.view_image(ref.T,  colormap="gray_r")
for f in path_list[1:]:
    #Apply transform
    path = f / "registration" 
    mov1 = fl.load(path / "mov_mapped.h5")
    viewer = napari.view_image(mov1.T,  colormap="gray_r")